In [1]:
import sys
sys.path.append("../src")
from dataset import ASVspoof2019Dataset

In [2]:
train_ds = ASVspoof2019Dataset(
    root_dir="../data/asvspoof2019/LA",
    split="train",
    max_len_sec=4.0
)

In [3]:
print("Total samples:", len(train_ds))

waveform, label, filename = train_ds[0]
print("Waveform shape:", waveform.shape)
print("Label:", label, "(0=bonafide, 1=spoof)")
print("Filename:", filename)

Total samples: 25380
Waveform shape: torch.Size([64000])
Label: 0 (0=bonafide, 1=spoof)
Filename: LA_T_1138215


In [4]:
# sanity check the label lookup matches what we confirmed manually earlier
waveform2, label2, filename2 = None, None, None
for w, l, fn in train_ds:
    if fn == "LA_T_1000137":
        waveform2, label2, filename2 = w, l, fn
        break
print("LA_T_1000137 label:", label2, "(should be 1=spoof, matches our earlier grep check)")

LA_T_1000137 label: 1 (should be 1=spoof, matches our earlier grep check)


In [5]:
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Model
import torch

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

model_name = "facebook/wav2vec2-base"
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)
wav2vec2 = Wav2Vec2Model.from_pretrained(model_name).to(device)
wav2vec2.eval()  # freeze in eval mode for now — we're not fine-tuning yet

print("Model loaded.")

/Users/dwibon/Desktop/sih25104-voice-detect/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: mps


Loading weights: 100%|████████████████████| 211/211 [00:00<00:00, 42852.90it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.bias             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded.


/Users/dwibon/Desktop/sih25104-voice-detect/venv/lib/python3.13/site-packages/huggingface_hub/file_download.py:755: UserWarning: Not enough free disk space to download the file. The expected file size is: 380.20 MB. The target location /Users/dwibon/.cache/huggingface/hub/models--facebook--wav2vec2-base/blobs only has 228.47 MB free disk space.
  warnings.warn(


In [6]:
waveform, label, filename = train_ds[0]
print("Raw waveform shape:", waveform.shape)

# Wav2Vec2FeatureExtractor expects a numpy array or list, handles normalization
inputs = feature_extractor(waveform.numpy(), sampling_rate=16000, return_tensors="pt")
input_values = inputs.input_values.to(device)
print("Input to model shape:", input_values.shape)

with torch.no_grad():
    outputs = wav2vec2(input_values)

hidden_states = outputs.last_hidden_state
print("Wav2Vec2 output shape:", hidden_states.shape)  # (batch, time_steps, hidden_dim)

# pool over time to get one fixed-size vector per clip
pooled = hidden_states.mean(dim=1)
print("Pooled embedding shape:", pooled.shape)  # should be (1, 768) for wav2vec2-base

Raw waveform shape: torch.Size([64000])
Input to model shape: torch.Size([1, 64000])
Wav2Vec2 output shape: torch.Size([1, 199, 768])
Pooled embedding shape: torch.Size([1, 768])


In [1]:
import sys
sys.path.append("../src")
import importlib
import extract_embeddings
importlib.reload(extract_embeddings)

import time
from torch.utils.data import Subset
from dataset import ASVspoof2019Dataset

# monkey-patch a temporary small-subset test
root_dir = "../data/asvspoof2019/LA"
ds = ASVspoof2019Dataset(root_dir, split="train", max_len_sec=4.0)
print("Full train set size:", len(ds))

# time just 5 batches (40 samples) to estimate full-run time
from model import Wav2Vec2Classifier
from torch.utils.data import DataLoader
import torch

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = Wav2Vec2Classifier(freeze_backbone=True).to(device)

def collate(batch):
    waveforms, labels, filenames = zip(*batch)
    waveforms_np = [w.numpy() for w in waveforms]
    inputs = model.feature_extractor(waveforms_np, sampling_rate=16000, return_tensors="pt", padding=True)
    return inputs.input_values, torch.tensor(labels), filenames

loader = DataLoader(ds, batch_size=16, shuffle=False, collate_fn=collate)

start = time.time()
n_batches_to_test = 5
with torch.no_grad():
    for i, (input_values, labels, filenames) in enumerate(loader):
        input_values = input_values.to(device)
        outputs = model.backbone(input_values)
        pooled = outputs.last_hidden_state.mean(dim=1)
        if i + 1 >= n_batches_to_test:
            break
elapsed = time.time() - start
per_batch = elapsed / n_batches_to_test
samples_per_sec = (n_batches_to_test * 16) / elapsed
est_full_train_time = len(ds) / samples_per_sec

print(f"Tested {n_batches_to_test} batches (batch_size=16) in {elapsed:.2f}s")
print(f"~{per_batch:.2f}s/batch, ~{samples_per_sec:.2f} samples/sec")
print(f"Estimated time to embed full train set ({len(ds)} samples): {est_full_train_time/60:.1f} minutes")

/Users/dwibon/Desktop/sih25104-voice-detect/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Full train set size: 25380


Loading weights: 100%|████████████████████| 211/211 [00:00<00:00, 42008.74it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.bias             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/Users/dwibon/Desktop/sih25104-voice-detect/venv/lib/python3.13/site-packages/huggingface_hub/file_download.py:755: UserWarning: Not enough free disk space to download the file. The expected file size is: 380.20 MB. The target location /Users/dwibon/.cache/huggingface/hub/models--facebook--wav2ve

Tested 5 batches (batch_size=16) in 10.92s
~2.18s/batch, ~7.33 samples/sec
Estimated time to embed full train set (25380 samples): 57.7 minutes
